![Token-Gated Language Models](https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/title.png)

[![GitHub](https://img.shields.io/badge/GitHub-TokenGate-181717?logo=github&logoColor=white)](https://github.com/token-gate-llms/TokenGate) [![arXiv](https://img.shields.io/badge/arXiv-2610.26754-B31B1B?logo=arxiv&logoColor=white)](https://arxiv.org/abs/2610.26754) [![Hugging Face paper](https://img.shields.io/badge/%F0%9F%A4%97-TokenGate%20paper-FFD21E)](https://huggingface.co/papers/2610.26754) [![X](https://img.shields.io/badge/X-Launch%20thread-000000?logo=x&logoColor=white)](https://x.com/lianapatel_/status/2112924355240099875) [![LinkedIn](https://img.shields.io/badge/LinkedIn-Post-0A66C2?logo=linkedin&logoColor=white)](https://lnkd.in/p/g3mj-QgZ)

## [0] Notebook overview

This notebook is a runnable introduction to TokenGate. You will:

1. **Set up** a needle-in-a-haystack task: hide a fact in a long document and ask for it.
2. **Run inference** with an ordinary model and a released TokenGate checkpoint, compare answers and prefill time at several token budgets, and inspect the tokens selected by the gate.
3. **Optionally train** a gate and a separate LoRA baseline on A100 GPUs, then compare them on held-out questions.

The default inference path runs on a free Colab T4; the training path needs A100 GPUs.

### What is TokenGate?

TokenGate is a small, drop-in component that extends existing trained models to enable end-to-end learnable token pruning over long-context inputs for fast and memory-efficient prefill.

*Prefill* is the work of reading the prompt before the first answer token.

### How token pruning works

A learned gate selects document tokens for later layers while keeping the question and instructions available.

![Animation: TokenGate selects document tokens after an early layer, so later layers process fewer tokens](https://raw.githubusercontent.com/token-gate-llms/TokenGate/0a287fbbd4bfeff6df566a0eae4b1e3134f21c28/assets/tokengate_explainer.gif)

### Published results

The [paper](https://arxiv.org/abs/2610.26754) reports the following matched-accuracy results for Qwen3-1.7B on A100 GPUs, against a separately LoRA-tuned model using full prefill. These are published benchmark numbers; the demo below measures your own run.

<p align="center"><img src="https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/results_figure.png" width="700" alt="Published TokenGate speed and memory results for NIAH and RepoBench"></p>


### The paper's Pareto frontier

The [paper figure](https://arxiv.org/html/2610.26754v1#S5) places accuracy on the vertical axis and prefill time or memory on the horizontal axis. Points toward the **upper left** answer as well while using less time or memory. Red shows TokenGate; blue is a separately LoRA-tuned model reading the full prompt; gray shows other methods. These A100 results are context for the smaller experiment you will run below.

<p align="center"><img src="https://raw.githubusercontent.com/token-gate-llms/TokenGate/0e7af59e4e8cbf7b6ad2db33f1875ce9eccf45f9/assets/tokengate_results_annotated.png" width="1000" alt="Accuracy versus time to first token and peak prefill memory across five tasks"></p>


## 1. Setup and information

We will use **needle-in-a-haystack (NIAH)**. One sentence about an activity in a city is planted among unrelated essays; the model must answer a question about that sentence. Choose one of three held-out examples: a fact near the beginning, middle, or end. The default runs the middle example only. At 16k, all three choices come from the frozen 150-row test partition, outside the training rows.

Choose an example, model, document length, token budgets, and your GPU runtime below, then install the repository API. To try another question, change **Example** and rerun from the prompt cell. A **token budget K** limits how many *document* tokens TokenGate passes to later layers; the question and instructions are always kept. Qwen3-1.7B at 16k on T4 is the default. Qwen3.5-2B runs through 8k on T4, while its 16k option needs an A100.

The inference cells follow the public API: prepare a prompt with `apply_chat_template`, restore the trained model with `load_model`, and call `generate`. The helpers handle test data, repeated measurements, and plots so those calls stay easy to see.


In [ ]:
#@title Demo settings { display-mode: "form" }
EXAMPLE = 'Middle (Hanoi)' #@param ['Front (Amman)', 'Middle (Hanoi)', 'End (Regina)']
MODEL = 'Qwen3-1.7B' #@param ['Qwen3-1.7B', 'Qwen3.5-2B']
DOC_TOKENS = '16k' #@param ['2k', '4k', '8k', '16k']
TOKEN_BUDGETS = '100, 500, 2000' #@param {type:"string"}
GPU_RUNTIME = 'T4' #@param ['T4', 'A100']

BASE_MODELS = {'Qwen3-1.7B':'Qwen/Qwen3-1.7B', 'Qwen3.5-2B':'Qwen/Qwen3.5-2B'}
CHECKPOINTS = {'Qwen3-1.7B':('liana313/tokengate-niah-qwen3-1.7b','3ef4d84ebd161a9700acced6e92f7cd5c50225ad'),
               'Qwen3.5-2B':('liana313/tokengate-niah-qwen3.5-2b','b1fe3e72b9896efc821fbfa02a19fcded4de6d77')}
EXAMPLE_INDEX = {'Front (Amman)': 0, 'Middle (Hanoi)': 1, 'End (Regina)': 2}[EXAMPLE]
BASE_MODEL = BASE_MODELS[MODEL]
DOC_TOKENS = int(DOC_TOKENS[:-1]) * 1024
assert MODEL != 'Qwen3.5-2B' or GPU_RUNTIME == 'A100' or DOC_TOKENS <= 8192, 'On a T4, Qwen3.5 supports up to 8k; choose an A100 for 16k.'
BUDGETS = tuple(int(x.strip()) for x in TOKEN_BUDGETS.split(',') if x.strip())
assert BUDGETS and len(set(BUDGETS)) == len(BUDGETS) and all(k > 0 for k in BUDGETS)
print(EXAMPLE, '|', BASE_MODEL, '|', DOC_TOKENS, 'document tokens | K =', BUDGETS, '| GPU:', GPU_RUNTIME)

In [ ]:
#@title Install and import the API { display-mode: "form" }
import gc, importlib, logging, os, subprocess, sys, warnings
os.environ['HF_HUB_VERBOSITY'] = 'error'
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='You are sending unauthenticated requests to the HF Hub.*')
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
assert GPU_RUNTIME in torch.cuda.get_device_name(0), f'Select a {GPU_RUNTIME} GPU in Colab: Runtime → Change runtime type.'
SOURCE = Path('/content/TokenGate') if Path('/content').is_dir() else Path.cwd()
if Path('/content').is_dir():
    if not SOURCE.exists():
        subprocess.run(['git','clone','-q','--branch','tokengate_demo',
                        'https://github.com/token-gate-llms/TokenGate',str(SOURCE)],check=True)
    else:
        subprocess.run(['git','-C',str(SOURCE),'fetch','origin','tokengate_demo'],check=True)
        subprocess.run(['git','-C',str(SOURCE),'switch','-C','tokengate_demo',
                        'origin/tokengate_demo'],check=True)
    local_head = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
    remote_head = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','origin/tokengate_demo'],text=True).strip()
    assert local_head == remote_head, 'The Colab checkout did not update to the latest demo branch.'
assert (SOURCE/'tokengate/demo.py').exists(), 'TokenGate demo helper is missing from this checkout.'
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(SOURCE),
                'transformers==5.5.0','matplotlib'],check=True)
sys.path.insert(0,str(SOURCE))
# A Colab runtime can keep the previous editable install in sys.modules.
for name in list(sys.modules):
    if name == 'tokengate' or name.startswith('tokengate.'):
        del sys.modules[name]
importlib.invalidate_caches()
from transformers import AutoTokenizer, AutoModelForCausalLM
if MODEL == 'Qwen3.5-2B':
    from transformers.models.qwen3_5.modeling_qwen3_5 import is_fast_path_available
    if not is_fast_path_available:
        logging.getLogger('transformers.models.qwen3_5.modeling_qwen3_5').setLevel(logging.ERROR)
        print('Qwen3.5: PyTorch DeltaNet fallback is active; baseline and gate timings include it.')
from huggingface_hub import snapshot_download
from tokengate import (apply_chat_template, split_state_span, load_model,
                       TokenGateConfig, TokenGateTrainer)
from tokengate.demo import (niah_examples, niah_prompt_text, niah_prompt,
                            plain_results, gate_results,
                            result_figures, show_kept_tokens, release_niah_rows,
                            release_niah_config_kwargs)
print('GPU:', torch.cuda.get_device_name(0))

## 2. Inference

We will give the *same full document and question* to the ordinary model and TokenGate. First, the notebook prints the selected question, its expected answer, and the prompt template. Then it runs both models and checks whether each answer contains the planted fact.

### Ordinary model: read the whole document

In the prompt template, `{state}` marks the document TokenGate may prune. `apply_chat_template(tokenizer, prompt, state=document)` inserts the document and prepares token IDs. The question and instructions sit outside `{state}`, so the gate cannot remove them. `split_state_span` removes the gate's markers to give the ordinary model the same text.

The ordinary model reads the complete prompt. Its answer is our first check that the fact is retrievable; its **time to first token (TTFT)** is the timing reference for the TokenGate plots.


In [ ]:
# Prepare the selected held-out NIAH example
samples = [niah_examples(SOURCE, DOC_TOKENS)[EXAMPLE_INDEX]]
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
prompt_templates = [niah_prompt_text(row) for row in samples]
print("Example question:", samples[0]['question'])
print("Expected answer:", samples[0]['answer'])
print("Prompt template:\n" + prompt_templates[0])
gate_prompts = [apply_chat_template(tok, template, state=row['body'])
                for template, row in zip(prompt_templates, samples)]
plain_prompts = [split_state_span(ids)[0] for ids in gate_prompts]
for row in samples:
    print(f"{row['city']:7s} needle at {row['depth_actual']:.0%} · {row['question']}")


In [ ]:
#@title Run the ordinary model { display-mode: "form" }
if GPU_RUNTIME == 'A100':
    assert torch.cuda.get_device_properties(0).total_memory >= 35*2**30, 'Use an A100-class GPU for single-pass prefill.'
dtype = torch.float16 if GPU_RUNTIME == 'T4' else torch.bfloat16
CHUNK_THRESHOLD = 4096  # Qwen3 on T4: chunk only above 4k document tokens
PLAIN_CHUNK_TOKENS = 1024 if GPU_RUNTIME == 'T4' and MODEL == 'Qwen3-1.7B' and DOC_TOKENS > CHUNK_THRESHOLD else None
print('Ordinary-model prefill:', f'cached {PLAIN_CHUNK_TOKENS}-token chunks (Qwen3/T4, above 4k)' if PLAIN_CHUNK_TOKENS else 'single pass')
plain_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype,
    attn_implementation='sdpa').cuda().eval()
plain = plain_results(plain_model, tok, plain_prompts, samples,
    chunk_tokens=PLAIN_CHUNK_TOKENS)
for row in plain:
    print(f"{row['city']:7s} correct={bool(row['correct'])} "
          f"TTFT={row['prefill_s']:.2f}s answer={row['answer']!r}")
del plain_model; gc.collect(); torch.cuda.empty_cache()

### Accelerate with TokenGate

Now `load_model` restores the base model, trained gate, and paired LoRA weight updates as one object. We call `gate.generate` on the selected question to show the API, then measure this question at each budget. The checkpoint stays the same as K changes.

`max_kept_prefill_tokens=K` limits *document* tokens passed to later layers. In the returned result, `answers` contains generated text, `n_state` counts document tokens presented to the gate, and `n_kept` counts all tokens kept for later layers, including the protected question and instructions. Smaller K can save work, but it may lose the fact, so compare answers as well as timing.


In [ ]:
# Load a checkpoint, try the API, then benchmark each token budget
repo_id, revision = CHECKPOINTS[MODEL]
checkpoint = snapshot_download(repo_id, revision=revision)
gate = load_model(checkpoint, gate_attn='sdpa' if GPU_RUNTIME == 'T4' else 'flex', dtype=dtype)
example_out = gate.generate(gate_prompts[0], tok, max_kept_prefill_tokens=BUDGETS[0],
                            max_new_tokens=64)
print(f"API example: K={BUDGETS[0]}, answer={example_out.answers[0]!r}")
print(f"Document tokens: {example_out.n_state[0]:,}; "
      f"total prompt tokens kept: {example_out.n_kept[0]:,}")
gated = gate_results(gate, tok, gate_prompts, samples, BUDGETS)
for row in gated:
    print(f"{row['city']:7s} K={str(row['budget']):5s} correct={bool(row['correct'])} "
          f"TTFT={row['prefill_s']:.2f}s retained={row['retained']:.1%} "
          f"answer={row['answer']!r}")

### Interpret the results

Read the generated answers first. The first plot shows **TTFT** for the ordinary model and each TokenGate budget; lower is faster. The second shows **speedup**: ordinary-model time divided by TokenGate time. A value of **2×** means TokenGate reached the first token in half the time. The short takeaway below the plots identifies the smallest tested K that answered this question, if there is one. This is one illustrative example, not an accuracy estimate.

These live plots compare TokenGate with the **unmodified base model**. The paper's figures above instead use a separately LoRA-tuned full-prefill baseline. On T4, the ordinary Qwen3 baseline also uses cached chunks **above 4k** to fit in memory; the paper measures single-pass prefill. Qwen3.5 uses a single pass through 8k on T4, and A100 uses one at every available length. The displayed T4 speedups therefore are not exact paper reproductions.


In [ ]:
#@title Prefill time and speedup { display-mode: "form" }
from IPython.display import display
for fig in result_figures(plain, gated,
    baseline_name='chunked base (T4)' if PLAIN_CHUNK_TOKENS else f'single-pass base ({GPU_RUNTIME})'):
    display(fig)

if not all(row['correct'] for row in plain):
    print('The ordinary model missed the fact; inspect its answer before comparing accuracy.')
passing = [k for k in BUDGETS
           if all(row['correct'] for row in gated if row['budget'] == k)]
if passing:
    print(f'For this question, K={min(passing)} is the smallest tested budget that retrieved the fact.')
else:
    print('No tested budget retrieved the fact; inspect the answers or try a larger K.')


### See what the gate selected

Finally, inspect the selected question at the smallest budget. Highlighted spans are document tokens that survived pruning; the question and instructions were protected throughout. Compare the highlighted text with the expected answer printed above. This shows *what* the gate kept, although the model's earlier layers may also have carried information from tokens it later dropped.


In [ ]:
#@title Which document tokens did the gate keep? { display-mode: "form" }
show_kept_tokens(gate, tok, gate_prompts[0], samples[0], budget=min(BUDGETS))

## 3. Optional: train and compare your own models

The results above use released weights. This optional A100 section trains a new gate and paired LoRA updates on 12,000 NIAH rows, using 50 validation rows and holding 150 rows out for testing. It also trains a separate LoRA-only model so the comparison uses the paper's kind of full-prefill baseline. Four A100s match the paper's batch layout; fewer GPUs take longer, and a full run can take hours.

To run it, switch to an A100 runtime and enable `RUN_TRAINING`. The code shows the public training pattern: put the recipe settings in `TokenGateConfig`, pass `article` and `answer` rows to `TokenGateTrainer`, and call `trainer.train()`. The final cell loads both trained models and compares their answers and prefill times on the selected held-out question at 16k. That quick check is not the full paper benchmark.


In [ ]:
# Train TokenGate and the matched LoRA baseline (A100 only)
RUN_TRAINING = False #@param {type:"boolean"}
if RUN_TRAINING:
    assert GPU_RUNTIME == 'A100', 'Choose A100 in Demo settings and switch to an A100 runtime.'
    n_gpus = torch.cuda.device_count()
    assert n_gpus in (1, 2, 4) and all('A100' in torch.cuda.get_device_name(i) for i in range(n_gpus)), 'Use 1, 2, or 4 visible A100s.'
    del gate
    gc.collect(); torch.cuda.empty_cache()
    work = Path('/content/tokengate-training') if Path('/content').is_dir() else Path.cwd()/'tokengate-training'
    work = work/MODEL
    data_root = work/'data'
    train_rows, eval_rows = release_niah_rows(SOURCE, work)
    recipe = release_niah_config_kwargs(SOURCE, BASE_MODEL, work/'gate', data_root, n_gpus)
    shown = ("base_model", "gate_layer", "learning_rate", "epochs",
             "max_steps", "k_eval", "lora_rank")
    print("TokenGateConfig:", {key: recipe[key] for key in shown})
    config = TokenGateConfig(**recipe)
    trainer = TokenGateTrainer(config, train_dataset=train_rows, eval_dataset=eval_rows, task='niah_kamradt')
    gate_checkpoint = Path(config.output_dir)/'gate_final.pt'
    if not gate_checkpoint.exists():
        if MODEL == 'Qwen3-1.7B':
            from kernels.fa2bias.keybias import available as fa2_available
            if not fa2_available():
                subprocess.run([sys.executable, str(SOURCE/'kernels/fa2bias/build.py')], check=True)
        trainer.train()
    del trainer, train_rows, eval_rows
    gc.collect()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft>=0.15'], check=True)
    baseline_dir = work/'lora_baseline'
    if not (baseline_dir/'adapter_model.safetensors').exists():
        baseline_steps = 10000 if MODEL == 'Qwen3-1.7B' else 18000
        subprocess.run([sys.executable, '-m', 'torch.distributed.run',
                        f'--nproc_per_node={n_gpus}', '--master_port=29501',
                        '-m', 'tokengate.demo_baseline', '--base-model', BASE_MODEL,
                        '--data-root', str(data_root), '--eval-split', str(config.eval_split),
                        '--output-dir', str(baseline_dir), '--max-steps', str(baseline_steps),
                        '--grad-accum', str(4//n_gpus)], check=True)
    print('Gate:', gate_checkpoint, '| LoRA baseline:', baseline_dir)
else:
    print('Optional full training skipped; inference above used the released checkpoint.')


In [ ]:
#@title Compare your trained gate with its LoRA baseline (A100 only) { display-mode: "form" }
if RUN_TRAINING:
    from peft import PeftModel
    benchmark_rows = [niah_examples(SOURCE, 16384)[EXAMPLE_INDEX]]
    pairs = [niah_prompt(tok, row) for row in benchmark_rows]
    trained_gate_prompts, baseline_prompts = map(list, zip(*pairs))
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, attn_implementation='sdpa')
    tuned_base = PeftModel.from_pretrained(base, str(baseline_dir)).merge_and_unload().cuda().eval()
    tuned_plain = plain_results(tuned_base, tok, baseline_prompts, benchmark_rows)
    print('LoRA baseline:', sum(r['correct'] for r in tuned_plain), '/', len(tuned_plain), 'correct')
    del tuned_base, base
    gc.collect(); torch.cuda.empty_cache()
    own_gate = load_model(str(gate_checkpoint), gate_attn='fa2' if MODEL == 'Qwen3-1.7B' else 'flex', dtype=torch.bfloat16)
    own = gate_results(own_gate, tok, trained_gate_prompts, benchmark_rows, BUDGETS)
    for k in BUDGETS:
        rows = [r for r in own if r['budget'] == k]
        print(f'K={k}: {sum(r["correct"] for r in rows)}/{len(rows)} correct')
    for fig in result_figures(tuned_plain, own, baseline_name='LoRA-tuned base (SDPA, A100)'):
        display(fig)
else:
    print('Run the optional training cell first.')
